In [0]:
INPUT_PATH = "/Volumes/platform_dev/iol_challenge/input/*.csv"
tabla_destino = "platform_dev.bronce_byma.operaciones_raw"

dbutils.widgets.text("full_load", "0")

full_load = int(dbutils.widgets.get("full_load"))

carga_full = (full_load == 1)

import logging
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("ingesta_operaciones")

In [0]:
try:
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(
            f"La tabla {tabla_destino} no existe. "
            "Se ejecutará el notebook de creación de tablas."
        )

        dbutils.notebook.run("../DDL/creacion_tablas", 0)

        carga_full = True

        logger.info(
            "Tablas y esquemas creados correctamente. "
            "Se realizará una carga full."
        )

    elif carga_full:
        logger.warning(
            f"Carga full forzada manualmente para {tabla_destino}."
        )

    else:
        logger.info(f"La tabla {tabla_destino} existe.")

        tiene_datos = spark.sql(
            f"SELECT 1 FROM {tabla_destino} LIMIT 1"
        ).take(1) != []

        if tiene_datos:
            logger.info(
                f"La tabla {tabla_destino} contiene datos. "
                "Se realizará una carga incremental."
            )
        else:
            carga_full = True

            logger.warning(
                f"La tabla {tabla_destino} está vacía. "
                "Se realizará una carga full."
            )

except Exception:
    logger.exception(
        f"Error al validar o crear la tabla {tabla_destino}"
    )
    raise

In [0]:
logger.info(f"Iniciando lectura del archivo: {INPUT_PATH}")

df = (
    spark.read
    .option("header", True)
    .option("delimiter", ",")
    .option("inferSchema", False)
    .csv(INPUT_PATH)
    .withColumn(
        "fecha_particion",
        F.to_date(
            F.from_utc_timestamp(
                F.to_timestamp("fecha"),
                "America/Argentina/Buenos_Aires")))
    .withColumn("_ingested_at", F.current_timestamp())
)

logger.info("Archivo leído correctamente y columnas técnicas generadas")

if not carga_full:
    # 1. Traemos la fila completa de control de cargas de forma segura
    fila_control = spark.sql(f"""
        SELECT ultima_fecha_procesada
        FROM platform_dev.bronce_byma.control_cargas
        WHERE proceso = '{tabla_destino}'
    """).first()

    # 2. Validamos defensivamente si el registro existe (Evita el crash por NoneType)
    if fila_control is not None and fila_control["ultima_fecha_procesada"] is not None:
        ultima_fecha = fila_control["ultima_fecha_procesada"]
        logger.info(
            f"Carga incremental para {tabla_destino}. "
            f"Última fecha procesada: {ultima_fecha}"
        )
    else:
        # Si está vacía, evitamos que rompa y le asignamos una fecha base histórica
        ultima_fecha = "1970-01-01"
        logger.warning(
            f"No se encontró registro de control para {tabla_destino} (Arranque en frío). "
            f"Se procesará desde la fecha base: {ultima_fecha}"
        )

    # 3. Aplicamos el filtro de deltas con la variable ya validada
    df = df.filter(
        F.col("fecha_particion") > F.lit(ultima_fecha)
    )

    logger.info(
        f"Filtro aplicado: fecha_particion > {ultima_fecha}"
    )

else:
    logger.info(
        f"Carga full para {tabla_destino}. "
        "No se aplica filtro por fecha."
    )

In [0]:
cantidad_registros = df.count()

try:
    if carga_full:
        logger.info(
            f"Carga full: iniciando escritura de {cantidad_registros} registros "
            f"en {tabla_destino}"
        )

        df.write \
            .format("delta") \
                .mode("overwrite") \
                    .saveAsTable(tabla_destino)

        logger.info(
            f"Carga full finalizada: {cantidad_registros} registros insertados"
        )

    else:
        if cantidad_registros == 0:
            logger.info("Carga incremental: no hay registros nuevos para insertar")

        else:
            logger.info(
                f"Carga incremental: iniciando escritura de "
                f"{cantidad_registros} registros en {tabla_destino}"
            )

            df.write \
                .format("delta") \
                    .mode("append") \
                        .saveAsTable(tabla_destino)

            logger.info(
                f"Carga incremental finalizada: "
                f"{cantidad_registros} registros insertados"
            )

except Exception:
    logger.exception(f"Error durante la escritura en {tabla_destino}")
    raise

In [0]:
ultima_fecha_cargada = df.agg(F.max("fecha_particion").alias("ultima_fecha")).first()["ultima_fecha"]

if ultima_fecha_cargada is not None:
    try:
        logger.info(
            f"Actualizando watermark del proceso {tabla_destino} "
            f"con fecha {ultima_fecha_cargada}"
        )

        spark.sql(f"""
        MERGE INTO platform_dev.bronce_byma.control_cargas AS target
        USING (
            SELECT
                '{tabla_destino}' AS proceso,
                DATE('{ultima_fecha_cargada}') AS ultima_fecha_procesada,
                current_timestamp() AS fecha_actualizacion
        ) AS source
        ON target.proceso = source.proceso

        WHEN MATCHED THEN UPDATE SET
            target.ultima_fecha_procesada = source.ultima_fecha_procesada,
            target.fecha_actualizacion = source.fecha_actualizacion

        WHEN NOT MATCHED THEN INSERT (
            proceso,
            ultima_fecha_procesada,
            fecha_actualizacion
        )
        VALUES (
            source.proceso,
            source.ultima_fecha_procesada,
            source.fecha_actualizacion
        )
        """)

        logger.info(
            f"Watermark actualizado correctamente: "
            f"proceso={tabla_destino}, fecha={ultima_fecha_cargada}"
        )

    except Exception:
        logger.exception(
            f"Error al actualizar watermark para {tabla_destino}"
        )
        raise

else:
    logger.warning(
        "No se actualizó la watermark porque no había registros nuevos"
    )

DUPLICADOS

In [0]:
df_duplicados = (
    spark.table("platform_dev.bronce_byma.operaciones_raw")
    .groupBy("id_transaccion")
    .agg(F.count("*").alias("count_dup"))
    .filter(F.col("count_dup") > 1)
    .withColumn("fecha_deteccion", F.current_timestamp())
)

df_duplicados.write \
    .format("delta") \
        .mode("append") \
            .saveAsTable("platform_dev.bronce_byma.calidad_duplicados")